# LangGraph: Tutorial sui Workflow e Agenti

Questo notebook esplora i pattern fondamentali di workflow con **LangGraph**, la libreria per costruire applicazioni LLM *stateful* basate su grafi.

| # | Pattern | Descrizione |
|---|---------|-------------|
| 1 | **LLM Potenziato** | LLM con output strutturato e strumenti |
| 2 | **Prompt Chaining** | Pipeline sequenziale step-by-step |
| 3 | **Parallelizzazione** | Fan-out / Fan-in: task in parallelo |
| 4 | **Routing** | Instradamento condizionale su path diversi |
| 5 | **Evaluator-Optimizer** | Loop genera → valuta → migliora |
| 6 | **Map-Reduce** | Elaborazione batch con aggregazione dinamica |
| 7 | **Orchestratore-Workers** | Decomposizione dinamica con LLM |
| 8 | **Agenti ReAct** | Agenti autonomi con uso di strumenti |

```bash
pip install langgraph langchain langchain-openai langchain-community python-dotenv
```

In [ ]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

set_llm_cache(SQLiteCache(database_path="cache/langchain.db"))

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

from langchain_openai import AzureChatOpenAI

model = AzureChatOpenAI(
    deployment_name="gpt-4o",
    azure_ad_token_provider=token_provider,
    temperature=0
)
print("Setup completato")

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from typing import TypedDict, Annotated, Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import operator

import langgraph
print("LangGraph version:", langgraph.__version__)

In [ ]:
# Articoli di notizie finanziarie di esempio
sample_articles = [
    "Nvidia shares head for record close as Wall Street shrugs off China concerns. "
    "Nvidia matched its intraday high as investors gain confidence the chipmaker will power through China restrictions.",

    "Sales of new homes tanked in May, pushing supply up to a 3-year high. "
    "New home sales dropped much more than expected, as high prices, high mortgage rates and weak consumer confidence ate into demand.",

    "Trump fumes over U.S. intelligence report on Iran strikes at NATO presser. "
    "Trump ordered U.S. strikes on Iranian nuclear sites as part of its support for Israel, then announced a ceasefire.",

    "Top oil CEOs sound the alarm as Israel-Iran standoff escalates. "
    "Some energy facilities have been hit in both Israel and Iran, raising concerns about supply disruptions.",

    "Gold outshines Treasurys, yen and Swiss franc as safe-haven assets show diverging performance amid global uncertainty."
]

print(f"Caricati {len(sample_articles)} articoli di esempio")

---

## Concetti Base di LangGraph

LangGraph modella i workflow come **grafi diretti** con tre componenti fondamentali:

```
    START
      |
   [node_a]   <-- funzione Python che legge state e restituisce aggiornamenti
      |
   [node_b]
      |
     END
```

- **State** – `TypedDict` con tutti i dati del workflow
- **Node** – funzione `(state) -> dict` con gli aggiornamenti
- **Edge** – connessione normale (`add_edge`) o condizionale (`add_conditional_edges`)
- **Reducer** – funzione che combina aggiornamenti multipli sullo stesso campo (es. `operator.add` per liste)

### Pattern base

```python
class MyState(TypedDict):
    input: str
    output: str

def my_node(state: MyState) -> dict:
    return {"output": "risultato"}   # aggiorna solo i campi necessari

builder = StateGraph(MyState)
builder.add_node("my_node", my_node)
builder.add_edge(START, "my_node")
builder.add_edge("my_node", END)
graph = builder.compile()

result = graph.invoke({"input": "ciao"})
```

In [ ]:
# --- Hello World: grafo minimo ---

class SimpleState(TypedDict):
    message: str
    response: str

def echo_node(state: SimpleState) -> dict:
    return {"response": f"Echo: {state['message'].upper()}"}

builder = StateGraph(SimpleState)
builder.add_node("echo", echo_node)
builder.add_edge(START, "echo")
builder.add_edge("echo", END)
simple_graph = builder.compile()

result = simple_graph.invoke({"message": "ciao LangGraph"})
print("Input: ", result["message"])
print("Output:", result["response"])

In [ ]:
# Visualizzazione della struttura del grafo
print(simple_graph.get_graph().draw_ascii())

---

## 1. LLM Potenziato (Augmented LLM)

Il pattern base: un LLM arricchito con **output strutturato**, **strumenti** o **memoria**.

```
  article
     |
[LLM + structured output]
     |
 NewsAnalysis (Pydantic)
```

`model.with_structured_output(Schema)` garantisce che la risposta sia un oggetto Pydantic valido,
eliminando il bisogno di parsing manuale del testo.

In [ ]:
class NewsAnalysis(BaseModel):
    sentiment: Literal["BULLISH", "BEARISH", "NEUTRAL"] = Field(
        description="Sentiment dal punto di vista finanziario"
    )
    key_asset: str = Field(description="Principale asset o settore citato")
    summary: str = Field(description="Riassunto in una frase")
    confidence: int = Field(description="Confidenza nella valutazione da 1 a 10")

structured_model = model.with_structured_output(NewsAnalysis)

class AugmentedState(TypedDict):
    article: str
    analysis: dict  # conterrà il NewsAnalysis serializzato

def augmented_llm_node(state: AugmentedState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Analizza questa notizia finanziaria e restituisci una valutazione strutturata.\n\n"
        "Notizia: {article}"
    )
    result = (prompt | structured_model).invoke({"article": state["article"]})
    return {"analysis": result.model_dump()}

builder = StateGraph(AugmentedState)
builder.add_node("analyze", augmented_llm_node)
builder.add_edge(START, "analyze")
builder.add_edge("analyze", END)
augmented_graph = builder.compile()

result = augmented_graph.invoke({"article": sample_articles[0]})
a = result["analysis"]
print(f"Sentiment:   {a['sentiment']}")
print(f"Asset:       {a['key_asset']}")
print(f"Summary:     {a['summary']}")
print(f"Confidence:  {a['confidence']}/10")

---

## 2. Prompt Chaining (Workflow Sequenziale)

I nodi vengono eseguiti **in sequenza**: ogni step usa l'output del precedente.

```
 article
    |
[summarize]          --> summary
    |
[classify_sentiment] --> sentiment
    |
[recommend]          --> recommendation
    |
   END
```

**Quando usarlo:** pipeline lineare dove ogni passo dipende dal precedente.

**In LangGraph:** `add_edge(node_a, node_b)` crea una dipendenza sequenziale.

In [ ]:
class ChainState(TypedDict):
    article: str
    summary: str
    sentiment: str
    recommendation: str

def summarize_node(state: ChainState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Riassumi questa notizia finanziaria in 2 frasi in italiano:\n{article}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"article": state["article"]})
    return {"summary": result}

def classify_sentiment_node(state: ChainState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Classifica il sentiment come BULLISH, BEARISH o NEUTRAL.\n"
        "Rispondi solo con una parola.\n\nNotizia: {summary}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"summary": state["summary"]})
    return {"sentiment": result.strip()}

def recommend_node(state: ChainState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Sei un trader. Dato il sentiment {sentiment}, scrivi una raccomandazione "
        "di trading in 2 frasi basandoti su:\n{summary}"
    )
    result = (prompt | model | StrOutputParser()).invoke({
        "sentiment": state["sentiment"],
        "summary": state["summary"]
    })
    return {"recommendation": result}

builder = StateGraph(ChainState)
builder.add_node("summarize", summarize_node)
builder.add_node("classify_sentiment", classify_sentiment_node)
builder.add_node("recommend", recommend_node)
builder.add_edge(START, "summarize")
builder.add_edge("summarize", "classify_sentiment")
builder.add_edge("classify_sentiment", "recommend")
builder.add_edge("recommend", END)
chain_graph = builder.compile()

print(chain_graph.get_graph().draw_ascii())

In [ ]:
result = chain_graph.invoke({"article": sample_articles[0]})
print("SOMMARIO:        ", result["summary"])
print("SENTIMENT:       ", result["sentiment"])
print("RACCOMANDAZIONE: ", result["recommendation"])

---

## 3. Parallelizzazione (Fan-out / Fan-in)

Task indipendenti vengono eseguiti **in parallelo** e i risultati vengono **aggregati**.

```
           article
              |
   +----------+----------+
   |          |          |
[summarize] [entities] [topic]
   |          |          |
   +----------+----------+
              |
           [merge]
              |
             END
```

**Quando usarlo:** task indipendenti da eseguire in parallelo per ridurre la latenza.

**In LangGraph:** basta aggiungere più `add_edge(START, node)`. Ogni nodo scrive su
un campo diverso, senza conflitti. Il nodo di merge aspetta che tutti completino.

In [ ]:
class ParallelState(TypedDict):
    article: str
    summary: str        # scritto da summarize
    entities: str       # scritto da extract_entities
    topic: str          # scritto da classify_topic
    final_report: str   # scritto da merge

def parallel_summarize(state: ParallelState) -> dict:
    prompt = ChatPromptTemplate.from_template("Riassumi in UNA frase chiave: {article}")
    return {"summary": (prompt | model | StrOutputParser()).invoke({"article": state["article"]})}

def extract_entities(state: ParallelState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Elenca le entita' principali (aziende, persone, luoghi) di questa notizia.\n"
        "Formato: nome (tipo), separati da virgola.\n\n{article}"
    )
    return {"entities": (prompt | model | StrOutputParser()).invoke({"article": state["article"]})}

def classify_topic(state: ParallelState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Classifica in UNA categoria: MACRO, TECNOLOGIA, ENERGIA, FINANZA, GEOPOLITICA\n"
        "Rispondi solo con la categoria.\n\n{article}"
    )
    return {"topic": (prompt | model | StrOutputParser()).invoke({"article": state["article"]}).strip()}

def merge_results(state: ParallelState) -> dict:
    report = (
        f"SOMMARIO:  {state['summary']}\n"
        f"ENTITA':   {state['entities']}\n"
        f"TOPIC:     {state['topic']}"
    )
    return {"final_report": report}

builder = StateGraph(ParallelState)
builder.add_node("summarize", parallel_summarize)
builder.add_node("extract_entities", extract_entities)
builder.add_node("classify_topic", classify_topic)
builder.add_node("merge", merge_results)

# Fan-out: tutti partono da START in parallelo
builder.add_edge(START, "summarize")
builder.add_edge(START, "extract_entities")
builder.add_edge(START, "classify_topic")

# Fan-in: tutti convergono in merge
builder.add_edge("summarize", "merge")
builder.add_edge("extract_entities", "merge")
builder.add_edge("classify_topic", "merge")
builder.add_edge("merge", END)

parallel_graph = builder.compile()
print(parallel_graph.get_graph().draw_ascii())

In [ ]:
result = parallel_graph.invoke({"article": sample_articles[3]})
print("--- RISULTATO PARALLELIZZAZIONE ---")
print(result["final_report"])

---

## 4. Routing (Instradamento Condizionale)

Il workflow viene instradato su **path diversi** in base al contenuto.

```
 article
    |
[classify]
    |
    +-- "MACRO"      --> [macro_analyst]  --> END
    +-- "TECNOLOGIA" --> [tech_analyst]   --> END
    +-- "ENERGIA"    --> [energy_analyst] --> END
    +-- altro        --> [general_analyst]--> END
```

**Quando usarlo:** quando hai specialisti/handler diversi per dominio.

**In LangGraph:** `add_conditional_edges(node, routing_fn)` dove `routing_fn`
restituisce il **nome del prossimo nodo** come stringa.

In [ ]:
class RoutingState(TypedDict):
    article: str
    category: str
    analysis: str

def classify_node(state: RoutingState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Classifica questa notizia in UNA categoria: MACRO, TECNOLOGIA, ENERGIA\n"
        "Rispondi solo con la categoria.\n\nNotizia: {article}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"article": state["article"]})
    return {"category": result.strip().upper()}

def route_by_category(state: RoutingState) -> str:
    cat = state["category"]
    if "MACRO" in cat:
        return "macro_analyst"
    elif "TECNOLOGIA" in cat or "TECH" in cat:
        return "tech_analyst"
    elif "ENERGIA" in cat or "ENERGY" in cat:
        return "energy_analyst"
    return "general_analyst"

def macro_analyst(state: RoutingState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Sei un analista macro. Analizza considerando tassi, inflazione e politica monetaria:\n{article}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"article": state["article"]})
    return {"analysis": f"[MACRO] {result}"}

def tech_analyst(state: RoutingState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Sei un analista tech. Analizza considerando valuazioni, crescita e innovazione:\n{article}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"article": state["article"]})
    return {"analysis": f"[TECH] {result}"}

def energy_analyst(state: RoutingState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Sei un analista energetico. Analizza considerando geopolitica e prezzi delle materie prime:\n{article}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"article": state["article"]})
    return {"analysis": f"[ENERGIA] {result}"}

def general_analyst(state: RoutingState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Analizza questa notizia finanziaria:\n{article}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"article": state["article"]})
    return {"analysis": f"[GENERALE] {result}"}

builder = StateGraph(RoutingState)
builder.add_node("classify", classify_node)
builder.add_node("macro_analyst", macro_analyst)
builder.add_node("tech_analyst", tech_analyst)
builder.add_node("energy_analyst", energy_analyst)
builder.add_node("general_analyst", general_analyst)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route_by_category)
builder.add_edge("macro_analyst", END)
builder.add_edge("tech_analyst", END)
builder.add_edge("energy_analyst", END)
builder.add_edge("general_analyst", END)

routing_graph = builder.compile()
print(routing_graph.get_graph().draw_ascii())

In [ ]:
for i, article in enumerate(sample_articles[:3]):
    result = routing_graph.invoke({"article": article})
    print(f"\n--- Articolo {i+1} ---")
    print(f"Categoria: {result['category']}")
    print(f"Analisi:   {result['analysis'][:200]}...")

---

## 5. Evaluator-Optimizer (Loop con Feedback)

Ciclo di **generazione → valutazione → miglioramento** che si ripete finché
la qualità supera una soglia o si raggiunge il numero massimo di iterazioni.

```
  article
     |
  [generate]  <---------+
     |                  | score < 7 e iter < 3
  [evaluate]            |
     |                  |
     +-- score >= 7 --> END
     |                  |
     +------------------+
```

**Quando usarlo:** vuoi garantire una qualita' minima nell'output con miglioramento iterativo.

**In LangGraph:** `add_conditional_edges` con funzione che restituisce il nome del nodo
oppure `END` per uscire dal ciclo.

In [ ]:
class EvalOptState(TypedDict):
    article: str
    recommendation: str
    score: int
    feedback: str
    iterations: int

def generate_node(state: EvalOptState) -> dict:
    feedback_str = ""
    if state.get("feedback"):
        feedback_str = f"\nFeedback dal valutatore (incorporalo): {state['feedback']}"

    prompt = ChatPromptTemplate.from_template(
        "Sei un analista finanziario. Scrivi una raccomandazione di trading professionale "
        "per questa notizia (max 4 frasi). Deve includere: asset, direzione, motivazione, rischio.{feedback}"
        "\n\nNotizia: {article}"
    )
    result = (prompt | model | StrOutputParser()).invoke({
        "article": state["article"],
        "feedback": feedback_str
    })
    return {
        "recommendation": result,
        "iterations": state.get("iterations", 0) + 1
    }

class EvalOutput(BaseModel):
    score: int = Field(description="Score da 1 a 10: chiarezza (3) + specificita' (4) + rischio (3)")
    feedback: str = Field(description="Suggerimento specifico per migliorare")

def evaluate_node(state: EvalOptState) -> dict:
    eval_model = model.with_structured_output(EvalOutput)
    prompt = ChatPromptTemplate.from_template(
        "Valuta questa raccomandazione di trading da 1 a 10.\n"
        "Criteri: chiarezza (max 3), specificita' (max 4), gestione del rischio (max 3).\n\n"
        "Notizia: {article}\n\nRaccomandazione: {recommendation}"
    )
    result = (prompt | eval_model).invoke({
        "article": state["article"],
        "recommendation": state["recommendation"]
    })
    print(f"  Iterazione {state.get('iterations', 1)}: score = {result.score}/10")
    return {"score": result.score, "feedback": result.feedback}

def should_continue(state: EvalOptState) -> str:
    if state["score"] >= 7 or state.get("iterations", 0) >= 3:
        return END
    return "generate"

builder = StateGraph(EvalOptState)
builder.add_node("generate", generate_node)
builder.add_node("evaluate", evaluate_node)
builder.add_edge(START, "generate")
builder.add_edge("generate", "evaluate")
builder.add_conditional_edges("evaluate", should_continue)
eval_opt_graph = builder.compile()

print(eval_opt_graph.get_graph().draw_ascii())

In [ ]:
result = eval_opt_graph.invoke({"article": sample_articles[0]})
print(f"\nScore finale:  {result['score']}/10")
print(f"Iterazioni:    {result['iterations']}")
print(f"\nRaccomandazione finale:\n{result['recommendation']}")

---

## 6. Map-Reduce

Elabora una **lista dinamica di items in parallelo** (map), poi **aggrega i risultati** (reduce).

```
 [art1, art2, art3, art4, art5]
          |
   +------+------+------+
   |      |      |      |  (Send API: fan-out dinamico)
 [w1]   [w2]   [w3]   [w4]
   |      |      |      |
   +------+------+------+
          |
     [aggregate]
          |
         END
```

**Quando usarlo:** lista di items da elaborare indipendentemente poi aggregare.

**In LangGraph:** si usa l'API `Send` per il fan-out dinamico. Il campo aggregato
usa `Annotated[list, operator.add]` come reducer (concatenazione di liste).

In [ ]:
class MapReduceState(TypedDict):
    articles: list[str]
    summaries: Annotated[list[str], operator.add]  # reducer: concatena le liste dei worker
    market_report: str

class WorkerState(TypedDict):
    article: str
    summaries: Annotated[list[str], operator.add]

def summarize_worker(state: WorkerState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Riassumi in UNA frase chiave per un trader (max 20 parole): {article}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"article": state["article"]})
    return {"summaries": [result.strip()]}

def fan_out_to_workers(state: MapReduceState) -> list:
    # Crea un Send per ogni articolo: ogni Send manda un worker su un articolo diverso
    return [Send("worker", {"article": article}) for article in state["articles"]]

def aggregate_node(state: MapReduceState) -> dict:
    bullets = "\n".join([f"- {s}" for s in state["summaries"]])
    prompt = ChatPromptTemplate.from_template(
        "Sei un analista di mercato. Sulla base di queste notizie:\n{bullets}\n\n"
        "Scrivi un report di mercato sintetico (3-4 frasi) con i temi principali "
        "e le possibili implicazioni per i trader."
    )
    result = (prompt | model | StrOutputParser()).invoke({"bullets": bullets})
    return {"market_report": result}

builder = StateGraph(MapReduceState)
builder.add_node("worker", summarize_worker)
builder.add_node("aggregate", aggregate_node)

# Fan-out dinamico: la funzione restituisce una lista di Send
builder.add_conditional_edges(START, fan_out_to_workers, ["worker"])
builder.add_edge("worker", "aggregate")
builder.add_edge("aggregate", END)

map_reduce_graph = builder.compile()
print(map_reduce_graph.get_graph().draw_ascii())

In [ ]:
result = map_reduce_graph.invoke({"articles": sample_articles})

print("--- SOMMARI INDIVIDUALI ---")
for i, s in enumerate(result["summaries"], 1):
    print(f"  {i}. {s}")

print(f"\n--- REPORT DI MERCATO ---")
print(result["market_report"])

---

## 7. Orchestratore-Workers

Un nodo **orchestratore** (LLM) decompone dinamicamente il task in sotto-task,
i **worker** li eseguono, un **sintetizzatore** aggrega i risultati.

```
   query
     |
[orchestrator]  --> genera task1, task2, task3
     |
  +--+--+
[w1] [w2] [w3]  (Send API: fan-out con task diversi)
  +--+--+
     |
[synthesizer]
     |
    END
```

**Differenza dal Map-Reduce:** i task sono **generati dall'LLM**, non predefiniti.

**Quando usarlo:** problemi aperti dove non sai a priori come scomporli.

In [ ]:
class OrchestratorState(TypedDict):
    query: str
    tasks: Annotated[list[str], operator.add]
    results: Annotated[list[str], operator.add]
    final_answer: str

class WorkerTaskState(TypedDict):
    task: str
    results: Annotated[list[str], operator.add]

def orchestrator_node(state: OrchestratorState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Sei un orchestratore. Decomponi questo task di analisi finanziaria in 3 sotto-task specifici.\n"
        "Elenca i sotto-task, uno per riga, ognuno che inizia con '-'.\n\n"
        "Task principale: {query}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"query": state["query"]})
    tasks = [
        line.strip().lstrip("-").strip()
        for line in result.strip().split("\n")
        if line.strip().startswith("-") and len(line.strip()) > 2
    ]
    print(f"Orchestratore ha generato {len(tasks)} task:")
    for t in tasks:
        print(f"  - {t[:80]}")
    return {"tasks": tasks}

def dispatch_to_workers(state: OrchestratorState) -> list:
    return [Send("worker_task", {"task": task, "results": []}) for task in state["tasks"]]

def worker_task_node(state: WorkerTaskState) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Completa questo sotto-task di analisi finanziaria in modo conciso e professionale:\n{task}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"task": state["task"]})
    return {"results": [f"[Task: {state['task'][:60]}]\n{result}"]}

def synthesizer_node(state: OrchestratorState) -> dict:
    combined = "\n\n".join(state["results"])
    prompt = ChatPromptTemplate.from_template(
        "Sintetizza questi risultati di analisi in una risposta finale coerente e professionale:\n\n"
        "{results}\n\nQuery originale: {query}"
    )
    result = (prompt | model | StrOutputParser()).invoke({
        "results": combined,
        "query": state["query"]
    })
    return {"final_answer": result}

builder = StateGraph(OrchestratorState)
builder.add_node("orchestrator", orchestrator_node)
builder.add_node("worker_task", worker_task_node)
builder.add_node("synthesizer", synthesizer_node)

builder.add_edge(START, "orchestrator")
builder.add_conditional_edges("orchestrator", dispatch_to_workers, ["worker_task"])
builder.add_edge("worker_task", "synthesizer")
builder.add_edge("synthesizer", END)

orchestrator_graph = builder.compile()
print(orchestrator_graph.get_graph().draw_ascii())

In [ ]:
result = orchestrator_graph.invoke({
    "query": "Analizza l'impatto delle tensioni geopolitiche in Medio Oriente "
             "sui mercati energetici e sulle strategie di portafoglio per un trader retail"
})
print("\n--- RISPOSTA FINALE ---")
print(result["final_answer"])

---

## 8. Agenti ReAct (Reason + Act)

Un agente che alterna **ragionamento** e **uso di strumenti** in modo autonomo.

```
   query
     |
  [agent]
     |
     +-- ha bisogno di un tool --> [tool] --> [agent]
     |                                     (loop)
     +-- risposta finale --> END
```

Il ciclo `Reason -> Act -> Observe` si ripete finche' l'agente ha tutte le informazioni.

**In LangGraph:** `create_react_agent(model, tools)` costruisce l'agente in modo semplice.
Gli strumenti sono funzioni Python decorate con `@tool`.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# Database mock di prezzi
MOCK_PRICES = {
    "NVDA": {"price": 140.5, "change_pct": 2.3, "sector": "Technology"},
    "OIL":  {"price": 82.5,  "change_pct": -1.2, "sector": "Energy"},
    "GOLD": {"price": 2380.0, "change_pct": 0.8, "sector": "Commodities"},
    "BTC":  {"price": 68000.0, "change_pct": 5.1, "sector": "Crypto"},
    "SPX":  {"price": 5450.0, "change_pct": 0.3, "sector": "Index"},
    "EUR":  {"price": 1.085,  "change_pct": -0.2, "sector": "FX"},
}

@tool
def get_asset_price(ticker: str) -> str:
    """Recupera prezzo corrente e variazione giornaliera di un asset (es: NVDA, OIL, GOLD, BTC, SPX, EUR)."""
    data = MOCK_PRICES.get(ticker.upper().strip())
    if not data:
        return f"Ticker non trovato. Disponibili: {', '.join(MOCK_PRICES.keys())}"
    sign = "+" if data["change_pct"] > 0 else ""
    return f"{ticker.upper()}: Prezzo={data['price']}, Variazione={sign}{data['change_pct']}%, Settore={data['sector']}"

@tool
def calculate_portfolio_value(holdings: str) -> str:
    """
    Calcola valore totale e rendimento giornaliero di un portafoglio.
    holdings: stringa nel formato 'TICKER1:quantita1,TICKER2:quantita2'
    Esempio: 'NVDA:10,GOLD:5,BTC:0.5'
    """
    total_value = 0.0
    total_change = 0.0
    lines = []
    for item in holdings.split(","):
        if ":" not in item:
            continue
        ticker, qty_str = item.strip().split(":", 1)
        ticker = ticker.strip().upper()
        qty = float(qty_str.strip())
        if ticker in MOCK_PRICES:
            p = MOCK_PRICES[ticker]["price"]
            c = MOCK_PRICES[ticker]["change_pct"]
            v = p * qty
            total_value += v
            total_change += v * c / 100
            lines.append(f"  {ticker}: {qty} x {p} = {v:.2f} ({c:+.1f}%)")
    if not lines:
        return "Nessun asset valido trovato."
    pct = total_change / total_value * 100 if total_value else 0
    return (
        "Portafoglio:\n" + "\n".join(lines) +
        f"\nValore totale: {total_value:,.2f}" +
        f"\nVariazione giornaliera: {total_change:+.2f} ({pct:+.2f}%)"
    )

@tool
def get_market_news_summary() -> str:
    """Restituisce un riassunto delle principali notizie di mercato del giorno."""
    return (
        "Notizie principali di oggi:\n"
        "1. Nvidia in rialzo su solidi dati AI (+2.3%)\n"
        "2. Petrolio in calo per accordo Iran-USA (-1.2%)\n"
        "3. Oro stabile come bene rifugio (+0.8%)\n"
        "4. Bitcoin sale su aspettative ETF europei (+5.1%)\n"
        "5. S&P 500 vicino ai massimi storici (+0.3%)"
    )

tools = [get_asset_price, calculate_portfolio_value, get_market_news_summary]
react_agent = create_react_agent(model, tools)

print("Agente ReAct creato con", len(tools), "strumenti:")
for t in tools:
    print(f"  - {t.name}")

In [ ]:
from langchain_core.messages import HumanMessage

# Test 1: recupero prezzi
result = react_agent.invoke({
    "messages": [HumanMessage(content="Qual e' il prezzo attuale di NVDA e di GOLD?")]
})
print("=== Test 1: Prezzi asset ===")
print(result["messages"][-1].content)

In [ ]:
# Test 2: analisi portafoglio + notizie
result = react_agent.invoke({
    "messages": [HumanMessage(
        content="Ho un portafoglio con 10 azioni NVDA, 5 once di GOLD e 0.5 BTC. "
                "Qual e' il valore totale e come sta andando oggi? "
                "Dammi anche un consiglio basato sulle notizie di mercato."
    )]
})
print("=== Test 2: Analisi portafoglio + raccomandazione ===")
print(result["messages"][-1].content)

---

## Riepilogo: Quando Usare Quale Pattern

| Pattern | Quando | Latenza | Complessita' |
|---------|--------|---------|-------------|
| **LLM Potenziato** | Output strutturato, singola chiamata | Bassa | Bassa |
| **Prompt Chaining** | Pipeline lineare, step dipendenti | Media | Bassa |
| **Parallelizzazione** | Task indipendenti, ridurre latenza | Bassa | Media |
| **Routing** | Specialisti diversi per dominio | Bassa | Media |
| **Evaluator-Optimizer** | Qualita' garantita, loop di miglioramento | Alta | Media |
| **Map-Reduce** | Lista dinamica di items + aggregazione | Media | Alta |
| **Orchestratore-Workers** | Decomposizione dinamica dei task | Alta | Alta |
| **Agenti ReAct** | Tool use autonomo, problemi aperti | Variabile | Alta |

## Concetti chiave da ricordare

- **I nodi restituiscono aggiornamenti**, non l'intero state
- **Reducers** (`Annotated[list, operator.add]`) gestiscono le scritture da nodi multipli
- **`Send` API** permette fan-out dinamico con state personalizzato per ogni worker
- **`add_conditional_edges`** con funzione che restituisce nome nodo o `END`
- **Checkpointing** (non trattato): `MemorySaver` o database per persistenza dello state

## Risorse
- [LangGraph Docs](https://langchain-ai.github.io/langgraph/)
- [LangGraph Tutorials](https://langchain-ai.github.io/langgraph/tutorials/)
- [LangGraph How-to Guides](https://langchain-ai.github.io/langgraph/how-tos/)